# Plot 2: CARROT vs binary routers on Routerbench

Binary baselines (route between GPT-4 and Mixtral-8x7B only): RouteLLM MF, Not-Diamond RoRF.
CARROT variants (route across all 11 models): CARROT-KNN, CARROT-RoBERTa.

Loads per-router predictions from `../data/routerbench/preds/` written by `carrot/train_and_infer.py`.

In [ ]:
import os, sys
sys.path.insert(0, '../carrot/')
import numpy as np
import matplotlib.pyplot as plt
from constants import LARGE_SMALL_MODELS
from utils import route, route_pairwise

os.makedirs('../plots', exist_ok=True)

DATASET = 'routerbench'
PREDS_DIR = f'../data/{DATASET}/preds'

meta = np.load(f'{PREDS_DIR}/meta.npy', allow_pickle=True).item()
models = meta['models']
Y_test = meta['Y_test']
C_test = meta['C_test']
small_ind = meta['small_model_ind']
large_ind = meta['large_model_ind']
print(f'{len(models)} models; test n={Y_test.shape[0]}')

In [ ]:
def load(name):
    path = f'{PREDS_DIR}/{name}.npy'
    if not os.path.exists(path):
        print(f'MISSING: {path}')
        return None
    return np.load(path, allow_pickle=True)

Y_hat = {m: load(f'Y_hat_{m}') for m in ['mf', 'rorf', 'roberta-binary', 'carrot-knn', 'carrot-roberta']}
C_hat = {m: load(f'C_hat_{m}') for m in ['carrot-knn', 'carrot-roberta']}

In [ ]:
mult = 100
curves = {}

# Binary routers (small-vs-large pairwise)
for name in ['mf', 'rorf', 'roberta-binary']:
    if Y_hat[name] is None:
        continue
    c, p = route_pairwise(np.asarray(Y_hat[name]).squeeze(), C_test, Y_test, large_ind, small_ind)
    curves[name] = (c, p)

# CARROT multi-model routers
for name in ['carrot-knn', 'carrot-roberta']:
    if Y_hat[name] is None or C_hat[name] is None:
        continue
    c, p = route(Y_hat[name], C_test, mult * C_hat[name], Y_test)
    curves[name] = (c, p)

print('Curves produced:', list(curves.keys()))

In [ ]:
from matplotlib.ticker import MaxNLocator

labels = {'mf': 'RouteLLM (MF)', 'rorf': 'Not-Diamond RoRF', 'roberta-binary': 'RouteLLM (RoBERTa)',
          'carrot-knn': 'CARROT (KNN)', 'carrot-roberta': 'CARROT (RoBERTa)'}
colors = {'mf': 'blue', 'rorf': 'green', 'roberta-binary': 'purple',
          'carrot-knn': 'red', 'carrot-roberta': 'orange'}
markers = ['o', 's', 'D', '^', 'v', 'p', '*', 'x', '+', 'h', 'H', 'd', '>', 'P']

LABEL_MODELS = {'gpt-4-1106-preview', 'zero-one-ai/Yi-34B-Chat',
                'gpt-3.5-turbo-1106', 'mistralai/mixtral-8x7b-chat'}

fig, ax = plt.subplots(1, 1, figsize=(4.3, 4.3))
for name, (c, p) in curves.items():
    ax.errorbar(c, p, c=colors[name], linestyle='--', linewidth=1, label=labels[name])

for i, m in enumerate(models):
    x, y = C_test[:, i].mean(0), Y_test[:, i].mean(0)
    ax.scatter([x], [y], marker=markers[i % len(markers)])
    if m in LABEL_MODELS:
        ax.annotate(m.split('/')[-1], (x, y), size=6)

ax.set_title('Routerbench — CARROT vs binary routers')
ax.set_xlabel('Cost Per Query, $')
ax.set_ylabel('Accuracy')
ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
ax.legend()
ax.grid(True)
fig.savefig('../plots/routerbench_binary.pdf', bbox_inches='tight')
plt.show()